In [1]:
import os
import hashlib
import json

import pandas as pd
import urllib.request

## **Descarga de datos**

In [2]:
def download_data(url, filename):
    """Downloads data from a URL and saves it to a file."""
    print(f"Downloading {url} to {filename}...")
    try:
        urllib.request.urlretrieve(url, filename)
        print(f" Downloaded {filename}")
    except Exception as e:
        print(f" Error downloading {filename}: {e}")
        raise

Datos de duración de viajes entre puntos en NYC

In [3]:
jan_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-01.parquet"
feb_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-02.parquet"

In [4]:
output_path = "data/processed"
data_path = "data"

In [5]:
os.makedirs(output_path, exist_ok=True)

In [6]:
download_data(jan_url, os.path.join(data_path, "jan.parquet"))
download_data(feb_url, os.path.join(data_path, "feb.parquet"))

 Error downloading data\jan.parquet: <urlopen error [Errno 11001] getaddrinfo failed>


URLError: <urlopen error [Errno 11001] getaddrinfo failed>

## **Carga de datos en pandas**

In [23]:
df_jan = pd.read_parquet(os.path.join(data_path, "jan.parquet"))
df_feb = pd.read_parquet(os.path.join(data_path, "feb.parquet"))

In [24]:
df_jan.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00


In [25]:
def duration_trip(df):
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    return df

In [26]:
df_jan = duration_trip(df_jan)
df_feb = duration_trip(df_feb)

In [27]:
df_jan.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,...,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75,11.016667
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,...,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00,6.766667
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,...,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00,6.333333
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,...,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00,5.816667
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,...,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00,5.966667


In [28]:
df_jan.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge',
       'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge',
       'duration'],
      dtype='object')

In [29]:
df_jan.trip_type.unique()

array([ 1.,  2., nan])

In [30]:
df_jan.drop(columns=['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'VendorID'], inplace=True)
df_feb.drop(columns=['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'VendorID'], inplace=True)

In [31]:
# save processed data
df_jan.to_parquet("data/processed/jan.parquet")
df_feb.to_parquet("data/processed/feb.parquet")

## **Metadata del dataset ("dataset version")**

En MLOps es importante poder identificar qué versión de datos usaste.

Aquí guardaremos un `metadata.json` simple con:
- cantidad de filas/columnas
- nombres de columnas
- checksum (hash) de los parquet procesados

In [32]:
def file_sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

metadata = {
    "jan": {
        "rows": int(df_jan.shape[0]),
        "cols": int(df_jan.shape[1]),
        "columns": list(df_jan.columns),
        "path": "data/processed/jan.parquet",
        "sha256": file_sha256("data/processed/jan.parquet"),
    },
    "feb": {
        "rows": int(df_feb.shape[0]),
        "cols": int(df_feb.shape[1]),
        "columns": list(df_feb.columns),
        "path": "data/processed/feb.parquet",
        "sha256": file_sha256("data/processed/feb.parquet"),
    },
}

metadata_path = "data/processed/metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

metadata

{'jan': {'rows': 65946,
  'cols': 18,
  'columns': ['store_and_fwd_flag',
   'RatecodeID',
   'PULocationID',
   'DOLocationID',
   'passenger_count',
   'trip_distance',
   'fare_amount',
   'extra',
   'mta_tax',
   'tip_amount',
   'tolls_amount',
   'ehail_fee',
   'improvement_surcharge',
   'total_amount',
   'payment_type',
   'trip_type',
   'congestion_surcharge',
   'duration'],
  'path': 'data/processed/jan.parquet',
  'sha256': '6f36242f4167e729e7c2b09068296502dd9e4be84a72283e61e72eb99096c803'},
 'feb': {'rows': 62574,
  'cols': 18,
  'columns': ['store_and_fwd_flag',
   'RatecodeID',
   'PULocationID',
   'DOLocationID',
   'passenger_count',
   'trip_distance',
   'fare_amount',
   'extra',
   'mta_tax',
   'tip_amount',
   'tolls_amount',
   'ehail_fee',
   'improvement_surcharge',
   'total_amount',
   'payment_type',
   'trip_type',
   'congestion_surcharge',
   'duration'],
  'path': 'data/processed/feb.parquet',
  'sha256': 'ab26e4d89ac73eaaad1a18e35ce780feb4d79028ed